# 第10回: Stochastic RNN - 確率的予測と不確実性のモデル化
**深層予測学習（Deep Predictive Learning）入門**

## 1. はじめに

### 確率的予測の動機

決定論的なモデル（通常のRNNやSARNN）は、入力に対して **1つの出力** しか生成できません。
しかし実世界のロボット動作には **不確実性** が伴います：
- 同じ状況でも複数の行動パターンが考えられる
- センサノイズによる入力の揺らぎ
- 環境の変化に対する応答の多様性

**Stochastic RNN** は、出力の **平均（mean）** と **分散（variance）** を同時に予測することで、
予測の不確実性をモデル化します。

| モデル | 出力 | 不確実性 |
|:---|:---|:---|
| 決定論的RNN | y = f(x) | なし |
| Stochastic RNN | μ, σ² = f(x) | あり |

分散が大きい箇所は「モデルが自信を持てない」状況を表し、
ロボット制御において安全性の判断に活用できます。

## 2. 環境セットアップ

In [ ]:
import torch
from tqdm.notebook import tqdm
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False
import os, json

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
torch.manual_seed(42)
np.random.seed(42)

## 3. SpatialSoftmax / InverseSpatialSoftmax

SARNNと同様に、画像から注意点座標を抽出するために使用します（第8回の復習）。

In [ ]:
class SpatialSoftmax(nn.Module):
    def __init__(self, width, height, temperature=1e-4):
        super().__init__()
        self.temperature = temperature
        pos_x, pos_y = np.meshgrid(
            np.linspace(0, 1, height), np.linspace(0, 1, width), indexing="xy")
        self.register_buffer("pos_x", torch.from_numpy(pos_x.reshape(-1)).float())
        self.register_buffer("pos_y", torch.from_numpy(pos_y.reshape(-1)).float())

    def forward(self, x):
        B, C, W, H = x.shape
        logit = x.reshape(B, C, -1)
        att = torch.softmax(logit / self.temperature, dim=-1)
        ex = torch.sum(self.pos_x * att, dim=-1, keepdim=True)
        ey = torch.sum(self.pos_y * att, dim=-1, keepdim=True)
        keys = torch.cat([ex, ey], -1).reshape(B, C, 2)
        return keys, att.reshape(B, C, W, H)


class InverseSpatialSoftmax(nn.Module):
    def __init__(self, width, height, heatmap_size=0.1):
        super().__init__()
        self.heatmap_size = heatmap_size
        pos_x, pos_y = np.meshgrid(
            np.linspace(0, 1, height), np.linspace(0, 1, width), indexing="xy")
        self.register_buffer("pos_xy",
            torch.from_numpy(np.stack([pos_x, pos_y], axis=0)).float())

    def forward(self, keys):
        d = torch.sum(
            torch.pow(self.pos_xy[None, None] - keys[:, :, :, None, None], 2.0), axis=2)
        return torch.exp(-d / self.heatmap_size)

print("SpatialSoftmax / InverseSpatialSoftmax 定義完了")

## 4. StochasticSARNN モデル

### 通常のSARNNとの違い

通常のSARNNでは関節角度の出力は **1つ**（`decoder_joint`）ですが、
StochasticSARNNでは **平均**（`decoder_joint_mean`）と **分散**（`decoder_joint_var`）の **2つ** を出力します。

分散は常に正の値である必要があるため、`torch.exp()` を適用します：

$$\sigma^2 = \exp(h_{\text{var}})$$

こ���により、線形層の出力がどのような値でも、分散は必ず正になります。

In [ ]:
class StochasticSARNN(nn.Module):
    def __init__(self, rec_dim, k_dim=5, joint_dim=7, temperature=1e-4,
                 heatmap_size=0.1, kernel_size=3, im_size=[64, 64]):
        super().__init__()
        self.k_dim = k_dim
        activation = nn.LeakyReLU(negative_slope=0.3)
        sub_im_size = [im_size[0] - 3*(kernel_size-1),
                       im_size[1] - 3*(kernel_size-1)]

        # Positional Encoder (SpatialSoftmax で注意点座標を抽出)
        self.pos_encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 0), activation,
            nn.Conv2d(16, 32, 3, 1, 0), activation,
            nn.Conv2d(32, k_dim, 3, 1, 0), activation,
            SpatialSoftmax(width=sub_im_size[0], height=sub_im_size[1],
                          temperature=temperature),
        )

        # Image Encoder (特徴マップ抽出)
        self.im_encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 0), activation,
            nn.Conv2d(16, 32, 3, 1, 0), activation,
            nn.Conv2d(32, k_dim, 3, 1, 0), activation,
        )

        # LSTM
        input_dim = joint_dim + k_dim * 2
        self.rec = nn.LSTMCell(input_dim, rec_dim)

        # ★ 確率的出力: 平均と分散の2つのデコーダ
        self.decoder_joint_mean = nn.Sequential(
            nn.Linear(rec_dim, joint_dim), activation)
        self.decoder_joint_var = nn.Sequential(
            nn.Linear(rec_dim, joint_dim), activation)

        # Point Decoder
        self.decoder_point = nn.Sequential(
            nn.Linear(rec_dim, k_dim * 2), activation)

        # Inverse Spatial Softmax
        self.issm = InverseSpatialSoftmax(
            width=sub_im_size[0], height=sub_im_size[1],
            heatmap_size=heatmap_size)

        # Image Decoder
        self.decoder_image = nn.Sequential(
            nn.ConvTranspose2d(k_dim, 32, 3, 1, 0), activation,
            nn.ConvTranspose2d(32, 16, 3, 1, 0), activation,
            nn.ConvTranspose2d(16, 3, 3, 1, 0), activation,
        )

        self.apply(self._weights_init)

    def _weights_init(self, m):
        if isinstance(m, nn.LSTMCell):
            nn.init.xavier_uniform_(m.weight_ih)
            nn.init.orthogonal_(m.weight_hh)
            nn.init.zeros_(m.bias_ih)
            nn.init.zeros_(m.bias_hh)
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, xi, xv, state=None):
        # Encode image
        im_hid = self.im_encoder(xi)
        enc_pts, _ = self.pos_encoder(xi)
        enc_pts = enc_pts.reshape(-1, self.k_dim * 2)

        # Concatenate and pass through LSTM
        hid = torch.cat([enc_pts, xv], -1)
        rnn_hid = self.rec(hid, state)

        # ★ 平均と分散を別々にデコード
        y_joint_mean = torch.tanh(self.decoder_joint_mean(rnn_hid[0]))
        y_joint_var = torch.exp(self.decoder_joint_var(rnn_hid[0]))  # 常に正

        # Point & Image decoding
        dec_pts = self.decoder_point(rnn_hid[0])
        dec_pts_in = dec_pts.reshape(-1, self.k_dim, 2)
        heatmap = self.issm(dec_pts_in)
        hid = torch.mul(heatmap, im_hid)
        y_image = self.decoder_image(hid)

        return y_image, y_joint_mean, y_joint_var, enc_pts, dec_pts, rnn_hid

# 動作確認
model = StochasticSARNN(rec_dim=50, k_dim=5, joint_dim=7).to(device)
dummy_img = torch.randn(2, 3, 64, 64, device=device)
dummy_joint = torch.randn(2, 7, device=device)
y_img, y_mean, y_var, enc_p, dec_p, _ = model(dummy_img, dummy_joint)
print(f"予測画像: {y_img.shape}")
print(f"関節平均: {y_mean.shape}")
print(f"関節分散: {y_var.shape} (min={y_var.min().item():.4f}, 常に正)")
print(f"エンコード注意点: {enc_p.shape}")
print(f"デコード注意点: {dec_p.shape}")
print(f"\nパラメータ数: {sum(p.numel() for p in model.parameters()):,}")

## 5. 誤差関数: Negative Log-Likelihood (NLL) Loss

### MSE vs NLL

通常のMSE損失は予測の **精度** のみを評価しますが、NLL損失は **精度と不確実性の両方** を評価します。

#### NLL Loss の数式

$$\mathcal{L}_{NLL} = \frac{1}{N}\sum_{i=1}^{N} \left[ \frac{\log(2\pi\sigma_i^2)}{2} + \frac{(y_i - \mu_i)^2}{2\sigma_i^2} \right]$$

- 第1項: 分散が大きいほどペナルティ（分散を小さくするよう学習）
- 第2項: 分散が小さいほど予測誤差のペナルティが大きい

つまり、モデルは「自信がある場所では正確に、自信がない場所では分散を大きくする」ことを学びます。

| 損失関数 | 出力 | 特徴 |
|:---|:---|:---|
| MSE | 平均のみ | 全てのデータを等しく扱う |
| NLL | 平均 + 分散 | データごとに不確実性を調整 |

In [ ]:
def calc_nll_loss(y_true, y_pred_mean, y_pred_var):
    """
    Negative Log-Likelihood Loss

    Args:
        y_true: 正解値 (batch, dim)
        y_pred_mean: 予測平均 (batch, dim)
        y_pred_var: 予測分散 (batch, dim), 常に正
    Returns:
        NLL loss (scalar)
    """
    constant_term = torch.log(2 * torch.pi * y_pred_var) / 2
    constant_term = torch.clamp(constant_term, min=1e-6)
    squared_error = ((y_true - y_pred_mean) ** 2) / (2 * y_pred_var)
    return torch.mean(constant_term + squared_error)


# NLL vs MSE の比較デモ
y_true = torch.tensor([[1.0, 2.0]])
y_mean = torch.tensor([[1.1, 1.8]])

# 分散が小さい場合: 予測に自信あり
y_var_small = torch.tensor([[0.01, 0.01]])
nll_small = calc_nll_loss(y_true, y_mean, y_var_small)

# 分散が大きい場合: 予測に自信なし
y_var_large = torch.tensor([[1.0, 1.0]])
nll_large = calc_nll_loss(y_true, y_mean, y_var_large)

mse = nn.MSELoss()(y_mean, y_true)

print(f"MSE Loss: {mse.item():.4f}")
print(f"NLL Loss (小分散 σ²=0.01): {nll_small.item():.4f}")
print(f"NLL Loss (大分散 σ²=1.0):  {nll_large.item():.4f}")
print("\n→ 分散が小さいと誤差に敏感、分散が大きいとペナルティが増加")

## 6. BPTT (Backpropagation Through Time)

StochasticSARNNの学習では、3種類の損失を組み合わせま���：

1. **画像損失** (MSE): 予測画像と正解画像の誤差
2. **関節損失** (NLL): 予測平均・分散と正解関節角度の負の対数尤度
3. **注意点損失** (MSE + annealing): エンコード注意点とデコード注意点の整合性

In [ ]:
class fullBPTTtrainer:
    def __init__(self, model, optimizer, loss_weights=[1.0, 1.0, 0.1], device="cpu"):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_weights = loss_weights
        self.device = device
        self.epoch_count = 0

    def process_epoch(self, images, joints, training=True):
        """
        Args:
            images: (batch, T, C, H, W)
            joints: (batch, T, joint_dim)
        """
        if training:
            self.model.train()
        else:
            self.model.eval()

        T = images.shape[1]
        state = None
        yi_list, ym_list, yv_list = [], [], []
        enc_list, dec_list = [], []

        self.optimizer.zero_grad()

        for t in range(T - 1):
            y_img, y_mean, y_var, enc_p, dec_p, state = self.model(
                images[:, t], joints[:, t], state)
            yi_list.append(y_img)
            ym_list.append(y_mean)
            yv_list.append(y_var)
            enc_list.append(enc_p)
            dec_list.append(dec_p)

        # Stack predictions
        yi_hat = torch.stack(yi_list, dim=1)
        ym_hat = torch.stack(ym_list, dim=1)
        yv_hat = torch.stack(yv_list, dim=1)

        # 1. Image loss (MSE)
        img_loss = nn.MSELoss()(yi_hat, images[:, 1:]) * self.loss_weights[0]

        # 2. Joint loss (NLL) ★
        joint_loss = calc_nll_loss(
            joints[:, 1:], ym_hat, yv_hat) * self.loss_weights[1]

        # 3. Point loss (MSE with annealing)
        pt_weight = min(self.epoch_count / 1000.0, 1.0) * self.loss_weights[2]
        pt_loss = nn.MSELoss()(
            torch.stack(dec_list[:-1]), torch.stack(enc_list[1:])) * pt_weight

        loss = img_loss + joint_loss + pt_loss

        if training:
            loss.backward()
            self.optimizer.step()
            self.epoch_count += 1

        return loss.item(), img_loss.item(), joint_loss.item(), pt_loss.item()

print("fullBPTTtrainer 定義完了")

## 7. Robomimic (PH) データを使った学習

Proficient-Human (PH) データセットを使用します。
PHは熟練者によるデモンストレーションで、一貫した品質のデータです。

ここではデモ用にダミーデータを生成します。

In [ ]:
# ダミー Robomimic PH データの生成
def create_dummy_ph_data(num_episodes=20, T=50, img_size=64, joint_dim=7):
    images = []
    joints = []
    for ep in range(num_episodes):
        t = np.linspace(0, 4*np.pi, T)
        # 関節角度: 滑らかな正弦波 + 小さいノイズ (PHなので高品質)
        j = np.stack([0.5*np.sin(t + i*0.5) for i in range(joint_dim)], axis=-1)
        j += np.random.normal(0, 0.02, j.shape)  # PH: 小さいノイズ
        joints.append(j.astype(np.float32))

        # 画像: 関節角度に基づいて円を描画
        imgs = []
        for step in range(T):
            img = np.ones((img_size, img_size, 3), dtype=np.float32) * 0.9
            cx = int((j[step, 0] * 0.4 + 0.5) * img_size)
            cy = int((j[step, 1] * 0.4 + 0.5) * img_size)
            cx = np.clip(cx, 3, img_size-4)
            cy = np.clip(cy, 3, img_size-4)
            img[cy-3:cy+3, cx-3:cx+3, 0] = 0.1  # Red circle
            img[cy-3:cy+3, cx-3:cx+3, 1] = 0.1
            img[cy-3:cy+3, cx-3:cx+3, 2] = 0.9
            imgs.append(img)
        images.append(np.array(imgs))

    return np.array(images), np.array(joints)

images_np, joints_np = create_dummy_ph_data()
print(f"画像: {images_np.shape}")  # (N, T, H, W, C)
print(f"関節: {joints_np.shape}")  # (N, T, joint_dim)

# PyTorch tensor に変換 (画像は NCHW 形式)
images_t = torch.from_numpy(images_np.transpose(0, 1, 4, 2, 3)).to(device)
joints_t = torch.from_numpy(joints_np).to(device)
print(f"\n画像テンソル: {images_t.shape}")
print(f"関節テンソル: {joints_t.shape}")

In [ ]:
# 学習
model = StochasticSARNN(rec_dim=50, k_dim=5, joint_dim=7).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trainer = fullBPTTtrainer(model, optimizer, loss_weights=[1.0, 0.5, 0.1], device=device)

num_epochs = 100
loss_history = {"total": [], "image": [], "joint": [], "point": []}

pbar = tqdm(range(num_epochs), desc="StochasticRNN学習")
for epoch in pbar:
    total, img_l, jnt_l, pt_l = trainer.process_epoch(images_t, joints_t, training=True)
    loss_history["total"].append(total)
    loss_history["image"].append(img_l)
    loss_history["joint"].append(jnt_l)
    loss_history["point"].append(pt_l)

    pbar.set_postfix(total=f"{total:.4f}", img=f"{img_l:.4f}", jnt=f"{jnt_l:.4f}")

print("学習完了！")

In [ ]:
# 損失曲線
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, key, title in zip(axes, ["image", "joint", "point"],
                           ["画像損失 (MSE)", "関節損失 (NLL)", "注意点損失"]):
    ax.plot(loss_history[key])
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 予測結果の可視化

### 8.1 入力画像 vs 予測画像、注意点

In [ ]:
# 推論
model.eval()
test_idx = 0
test_images = images_t[test_idx:test_idx+1]  # (1, T, C, H, W)
test_joints = joints_t[test_idx:test_idx+1]  # (1, T, joint_dim)

pred_imgs, pred_means, pred_vars = [], [], []
enc_pts_list, dec_pts_list = [], []
state = None

with torch.no_grad():
    for t in range(test_images.shape[1] - 1):
        y_img, y_mean, y_var, enc_p, dec_p, state = model(
            test_images[:, t], test_joints[:, t], state)
        pred_imgs.append(y_img[0].cpu().numpy().transpose(1, 2, 0))
        pred_means.append(y_mean[0].cpu().numpy())
        pred_vars.append(y_var[0].cpu().numpy())
        enc_pts_list.append(enc_p[0].cpu().numpy())
        dec_pts_list.append(dec_p[0].cpu().numpy())

pred_means = np.array(pred_means)
pred_vars = np.array(pred_vars)

# 画像比較 (5フレーム)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
frames = np.linspace(0, len(pred_imgs)-1, 5, dtype=int)
for i, f in enumerate(frames):
    gt = test_images[0, f+1].cpu().numpy().transpose(1, 2, 0)
    axes[0, i].imshow(np.clip(gt, 0, 1))
    axes[0, i].set_title(f"入力 t={f+1}")
    axes[0, i].axis("off")

    pred = np.clip(pred_imgs[f], 0, 1)
    axes[1, i].imshow(pred)
    # 注意点をオーバーレイ
    pts = dec_pts_list[f].reshape(-1, 2)
    axes[1, i].scatter(pts[:, 0]*64, pts[:, 1]*64, c="red", s=30, marker="x")
    axes[1, i].set_title(f"予測 t={f+1}")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("正解", fontsize=12)
axes[1, 0].set_ylabel("予測+注意点", fontsize=12)
plt.suptitle("入力画像 vs 予測画像（赤×: 予測注意点）", fontsize=14)
plt.tight_layout()
plt.show()

### 8.2 関節角度の予測（平均 ± 標準偏差）

確率的モデルの最大の利点は、**予測の信頼区間** を可視化できることです。
平均 ± 標準偏差の帯を表示することで、モデルがどこで不確実なのかが一目で分かります。

In [ ]:
# 関節角度: 平均 ± 標準偏差
gt_joints = test_joints[0, 1:].cpu().numpy()
pred_std = np.sqrt(pred_vars)  # 標準偏差 = √分散

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
joint_names = [f"Joint {i}" for i in range(7)]

for i in range(7):
    ax = axes[i]
    t = np.arange(len(pred_means))

    # 正解
    ax.plot(t, gt_joints[:, i], "b-", label="正解", linewidth=1.5)
    # 予測平均
    ax.plot(t, pred_means[:, i], "r--", label="予測平均", linewidth=1.5)
    # ± 標準偏差の帯
    ax.fill_between(t,
                    pred_means[:, i] - pred_std[:, i],
                    pred_means[:, i] + pred_std[:, i],
                    alpha=0.3, color="red", label="±σ")
    ax.set_title(joint_names[i])
    ax.set_xlabel("Time step")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)

# 空のサブプロットを非表示
for i in range(7, 9):
    axes[i].set_visible(False)

plt.suptitle("関節角度の予測（平均 ± 標準偏差）", fontsize=14)
plt.tight_layout()
plt.show()

### 8.3 分散の時間変化

分散が大きい箇所はモデルの不確実性が高い場面を示します。

In [ ]:
# 分散の時間変化
fig, ax = plt.subplots(figsize=(12, 4))
for i in range(7):
    ax.plot(pred_vars[:, i], label=f"Joint {i}", alpha=0.7)
ax.set_xlabel("Time step")
ax.set_ylabel("予測分散 σ²")
ax.set_title("各関節の予測分散の時間変化")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 全関節の平均分散
mean_var = pred_vars.mean(axis=1)
print(f"平均分散の範囲: {mean_var.min():.4f} ~ {mean_var.max():.4f}")
print(f"最も不確実なタイムステップ: t={mean_var.argmax()}")

### 演習1: StochasticSARNNのforward関数を実装せよ

以下のStochasticSARNNクラスのforward関数を完成させ��ください。
通常のSARNNとの違いは、関節角度の出力が **平均と分散の2つ** になる点です��
分散には `torch.exp()` を適用して非負性を保証してください。

In [ ]:
class MyStochasticSARNN(nn.Module):
    def __init__(self, rec_dim=50, k_dim=5, joint_dim=7, im_size=[64,64]):
        super().__init__()
        self.k_dim = k_dim
        act = nn.LeakyReLU(0.3)
        sub = [im_size[0]-6, im_size[1]-6]

        self.pos_encoder = nn.Sequential(
            nn.Conv2d(3,16,3,1,0), act, nn.Conv2d(16,32,3,1,0), act,
            nn.Conv2d(32,k_dim,3,1,0), act,
            SpatialSoftmax(sub[0], sub[1]))
        self.im_encoder = nn.Sequential(
            nn.Conv2d(3,16,3,1,0), act, nn.Conv2d(16,32,3,1,0), act,
            nn.Conv2d(32,k_dim,3,1,0), act)
        self.rec = nn.LSTMCell(joint_dim + k_dim*2, rec_dim)
        self.dec_mean = nn.Sequential(nn.Linear(rec_dim, joint_dim), act)
        self.dec_var = nn.Sequential(nn.Linear(rec_dim, joint_dim), act)
        self.dec_pt = nn.Sequential(nn.Linear(rec_dim, k_dim*2), act)
        self.issm = InverseSpatialSoftmax(sub[0], sub[1])
        self.dec_img = nn.Sequential(
            nn.ConvTranspose2d(k_dim,32,3,1,0), act,
            nn.ConvTranspose2d(32,16,3,1,0), act,
            nn.ConvTranspose2d(16,3,3,1,0), act)

    def forward(self, xi, xv, state=None):
        # TODO: ここを実装
        pass

# テスト
# m = MyStochasticSARNN().to(device)
# y_img, y_mean, y_var, enc_p, dec_p, state = m(dummy_img, dummy_joint)

<details>
<summary><b>回答例を表示</b></summary>

```python
def forward(self, xi, xv, state=None):
    im_hid = self.im_encoder(xi)
    enc_pts, _ = self.pos_encoder(xi)
    enc_pts = enc_pts.reshape(-1, self.k_dim * 2)
    hid = torch.cat([enc_pts, xv], -1)

    rnn_hid = self.rec(hid, state)

    # ★ 平均と分散を別々にデコード
    y_mean = torch.tanh(self.dec_mean(rnn_hid[0]))
    y_var = torch.exp(self.dec_var(rnn_hid[0]))  # 常に正

    dec_pts = self.dec_pt(rnn_hid[0])
    dec_pts_in = dec_pts.reshape(-1, self.k_dim, 2)
    heatmap = self.issm(dec_pts_in)
    hid = torch.mul(heatmap, im_hid)
    y_image = self.dec_img(hid)

    return y_image, y_mean, y_var, enc_pts, dec_pts, rnn_hid
```

</details>

### 演習2: NLL Loss関数を実装し、MSEとの違いを確認せよ

Negative Log-Likelihood Loss を自分で実装してください。
さらに、異なる分散値でのNLL損失の変化をプロットし、MSEとの違いを考察してください。

In [ ]:
def my_nll_loss(y_true, y_mean, y_var):
    # TODO: NLL Lossを実装
    pass

# テスト: 同じ予測誤差、異なる分散でNLLを比較
# variances = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
# ...

<details>
<summary><b>回答例を表示</b></summary>

```python
def my_nll_loss(y_true, y_mean, y_var):
    const = torch.log(2 * torch.pi * y_var) / 2
    const = torch.clamp(const, min=1e-6)
    sq_err = ((y_true - y_mean)**2) / (2 * y_var)
    return torch.mean(const + sq_err)

# 比較: 同じ誤差、異なる分散
y_true = torch.tensor([[1.0]])
y_mean = torch.tensor([[1.2]])  # 誤差 = 0.2

variances = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
nll_values = [my_nll_loss(y_true, y_mean, torch.tensor([[v]])).item() for v in variances]

mse_val = nn.MSELoss()(y_mean, y_true).item()

plt.figure(figsize=(8, 4))
plt.plot(variances, nll_values, 'bo-', label='NLL Loss')
plt.axhline(y=mse_val, color='r', linestyle='--', label=f'MSE Loss = {mse_val:.4f}')
plt.xlabel('分散 σ²')
plt.ylabel('Loss')
plt.title('NLL Loss vs MSE Loss（同じ予測誤差）')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
```

</details>

### 演習3: 予測の不確実性を可視化し、どのような場面で分散が大きくなるか考察せよ

学習済みモデルの予測分散を各関節・各タイムステップで可視化し、
どのような状況でモデルの不確実性が高くなるか考察してください。

ヒント: ヒートマップ表示が分かりやすいです。

In [ ]:
# ヒートマップで分散を可視化
# pred_vars の shape は (T, joint_dim)
# TODO: plt.imshow でヒートマップを作成

<details>
<summary><b>回答例を表示</b></summary>

```python
# ヒートマップで分散を可視化
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pred_vars.T, aspect='auto', cmap='hot', interpolation='nearest')
ax.set_xlabel("Time step")
ax.set_ylabel("Joint index")
ax.set_yticks(range(7))
ax.set_yticklabels([f"Joint {i}" for i in range(7)])
ax.set_title("予測分散のヒートマップ（明るい = 不確実性が高い）")
plt.colorbar(im, ax=ax, label="σ²")
plt.tight_layout()
plt.show()

# 考察
print("考察:")
print("- 動作の切り替わり点（方向転換）付近で分散が大きくなる傾向")
print("- 複数のパターンが考えられる場面で不確実性が高い")
print("- 学習データが少ない領域で分散が増加")
```

</details>

---
## まとめ

| 項目 | 内容 |
|:---|:---|
| **モデル** | StochasticSARNN: 平均 + 分散を出力 |
| **損失関数** | NLL Loss: 精度と不確実性を同時に最適化 |
| **分散の保証** | `torch.exp()` で常に正の値 |
| **可視化** | 平均±σの帯表示で信頼区��を表現 |
| **応用** | 不確実性の高い場面の検出、安全なロボット制御 |

次回: 第11回 Diffusion Policy → 拡散モデルによる行動生成